# Content-based Filtering

Recommending items based on their features and a user's taste profile — no other users involved.

## Imports and Configuration

In [1]:
import sys
sys.path.append('../../datasets') # two levels up from the notebook

In [51]:
import pandas as pd
from data_loader import DataLoader

## Load and Explore the Data

In [93]:
loader = DataLoader()

ratings_df = loader.load("ml-latest-small/ratings.csv")
movies_df = loader.load("ml-latest-small/movies.csv")

In [8]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [86]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [87]:
print(f"Ratings: {ratings_df.shape[0]} rows")
print(f"Movies: {movies_df.shape[0]} unique")
print(f"Users: {ratings_df["userId"].nunique()} unique")

Ratings: 100836 rows
Movies: 9742 unique
Users: 610 unique


## Prepare Movie data

Split the values in the Genres column into a list of Genres to simplify future use.

In [94]:
movies_df["genres"] = movies_df["genres"].apply(lambda x: x.split("|"))
movies_df.head(2)

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"


Iterate through movies_df, then append the movie genres as columns of 1s or 0s.  
1 if that column contains movies in the genre at the present index and 0 if not.

In [104]:
movies_with_genres = movies_df.copy(deep=True)

for index, row in movies_df.iterrows():
    for genre in row["genres"]:
        movies_with_genres.at[index, genre] = 1

movies_with_genres.head(3)

,movieId,title,genres,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1.0,1.0,1.0,1.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1.0,NaN,1.0,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",NaN,NaN,NaN,1.0,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Fill in the NaN values with 0 to show that a movie doesn't have that column's genre

In [103]:
movies_with_genres = movies_with_genres.fillna(0)
movies_with_genres.head(2)

,movieId,title,genres,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Prepare Rating Data

In [105]:
# We do not need the timestamp column
ratings_df.drop(columns="timestamp", inplace=True)
ratings_df.head(2)

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0


In [106]:
# Check for null values
ratings_df.isna().sum()

userId     0
movieId    0
rating     0
dtype: int64